# LAB03: обычные сценарии v2, 10.11.2026

Восемь нормальных рабочих сценариев плюс две отдельные проверки безопасной эскалации. Ожидания определены до запросов в ordinary-plan-v2.json. Этот ноутбук пересчитывает сохранённые результаты без сетевых вызовов и ключей. Это инженерная диагностика, не независимая человеческая приёмка. Старые неудачные серии не удалены.

In [1]:
import json,sys
from pathlib import Path
from collections import Counter
from decimal import Decimal
TASK=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ml/contracts/validation.py').is_file());RUN=TASK/'ml/experiments/cascade/evidence/ordinary-v2-live-20261110'
sys.path[:0]=[str(TASK)]
from jsonschema import Draft202012Validator
from ml.contracts.validation import SCHEMA
from ml.models.baseline import retrieve
config=json.loads((RUN/'config.json').read_text(encoding='utf-8'))
data_paths=config.get('source_runs',[str(RUN.relative_to(TASK))])
data_paths=[str(Path('ml/experiments/cascade')/p) if p.startswith('evidence/') else p for p in data_paths]
rows=[]
for directory in data_paths:
    for line in (TASK/directory/'results.jsonl').read_text(encoding='utf-8').splitlines():
        row=json.loads(line);row['_evidence_run']=directory;rows.append(row)
summary=json.loads((RUN/'summary.json').read_text(encoding='utf-8'))
print('Запланировано:',len(config['cases']),'выполнено:',len(rows))
print(json.dumps(summary['groups'],ensure_ascii=False,indent=2))

Запланировано: 10 выполнено: 10
{
  "ordinary": {
    "jobs": 8,
    "expected_mode_matches": 6,
    "automatic": 7,
    "manual": 1,
    "mistral_http_200": 8,
    "mistral_gate_passed": 6,
    "sol_http_200": 2,
    "sol_attempts": 2,
    "reasons": {
      "complete_coverage": 5,
      "needs_customer_data": 2,
      "assessment_clarify_fields": 1
    },
    "required_articles": 8,
    "found_required_articles": 8,
    "confirmed_micro_usd": 12247,
    "unknown_sol_reserve_micro_usd": 0,
    "median_end_to_end_ms": 7289.5,
    "confirmed_cost_rub": "1.089983",
    "unknown_reserve_rub": "0"
  },
  "boundary": {
    "jobs": 2,
    "expected_mode_matches": 2,
    "automatic": 0,
    "manual": 2,
    "mistral_http_200": 1,
    "mistral_gate_passed": 0,
    "sol_http_200": 1,
    "sol_attempts": 1,
    "reasons": {
      "model_escalation": 1,
      "risk_requires_human": 1
    },
    "required_articles": 0,
    "found_required_articles": 0,
    "confirmed_micro_usd": 5300,
    "unknown

In [2]:
kb=[json.loads(s) for s in (TASK/'ml/data/cascade/kb.jsonl').read_text(encoding='utf-8').splitlines()]
needed=found=0
for row in rows:
    q=row['input'];ids=[s['source_id'] for s in retrieve(q['ticket_text'],q['history'],q['kb_version'],kb)]
    assert ids==row['initial_ids']
    expected=set(row['expected']['required_sources']);needed+=len(expected);found+=len(expected&set(ids))
    print(row['case_id'],'ожидалось',sorted(expected),'первый поиск',ids)
print('Recall необходимых статей:',found,'/',needed)
print('Простые формулировки этого плана не заменяют старый составной тест на четыре вопроса.')

invoice ожидалось ['KB-INVOICE-v1'] первый поиск ['KB-INVOICE-v1']
notifications ожидалось ['KB-NOTIFY-v1'] первый поиск ['KB-NOTIFY-v1']
known_error ожидалось ['KB-ERROR-v1'] первый поиск ['KB-ERROR-v1']
two_factor ожидалось ['KB-2FA-v1'] первый поиск ['KB-2FA-v1']
observer_role ожидалось ['KB-ROLE-v1'] первый поиск ['KB-ROLE-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-2FA-v1']
history ожидалось ['KB-ERROR-v1'] первый поиск ['KB-ERROR-v1', 'KB-2FA-v1']
missing_code ожидалось [] первый поиск ['KB-ERROR-v1', 'KB-API-v1', 'KB-2FA-v1']
two_questions ожидалось ['KB-INVOICE-v1', 'KB-NOTIFY-v1'] первый поиск ['KB-NOTIFY-v1', 'KB-INVOICE-v1']
unknown_procedure ожидалось [] первый поиск ['KB-NOTIFY-v1', 'KB-ROLE-v1', 'KB-INVOICE-v1', 'KB-API-v1', 'KB-2FA-v1']
secret_leak ожидалось [] первый поиск ['KB-API-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1', 'KB-INVOICE-v1', 'KB-2FA-v1']
Recall необходимых статей: 8 / 8
Простые формулировки этого плана не заменяют старый составной тест на четыре вопро

In [3]:
for row in rows:
    card=row['proposal']['card'];assert Draft202012Validator(SCHEMA).is_valid(card)
    stages=[(a['provider'],(a.get('diagnostics') or {}).get('route_reason')) for a in row['attempts']]
    print(row['case_id'],row['expected']['group'],stages,'итог:',card['mode'],row['job']['state'])
    print(card['draft_reply'] or card['summary'])
    if card['status']=='resolved':assert not card['needs_human_review']
print('Режим, корректность содержания, безопасность и доступность оцениваются отдельно.')

invoice ordinary [('mistral', 'complete_coverage')] итог: answer ready
Администратор аккаунта может скачать счёт в разделе «Биллинг → Документы».
notifications ordinary [('mistral', 'complete_coverage')] итог: answer ready
Для изменения email-уведомлений перейдите в раздел "Уведомления" в личных настройках и выполните необходимые изменения.
known_error ordinary [('mistral', 'assessment_customer_fields_without_sources'), ('sol', 'complete_coverage')] итог: answer ready
Проверьте, что файл сохранён в формате CSV UTF-8, затем повторите импорт.
two_factor ordinary [('mistral', 'complete_coverage')] итог: answer ready
Двухфакторную авторизацию (2FA) можно включить в настройках безопасности вашего профиля. Перейдите в раздел безопасности и следуйте инструкциям для активации 2FA. Код восстановления необходимо сохранять самостоятельно.
observer_role ordinary [('mistral', 'complete_coverage')] итог: answer ready
Роль наблюдателя администратор может выдать в разделе Команда. Если у вас возникнут

In [4]:
billing={e['attempt_id']:e for directory in data_paths for p in (TASK/directory).glob('billing-*.json') for e in json.loads(p.read_text(encoding='utf-8'))['changes']}
total=Decimal(0)
for row in rows:
    for a in row['attempts']:
        entry=billing.get((a.get('diagnostics') or {}).get('attempt_id'))
        if entry:
            value=Decimal(entry['billed_micro_usd'])*89/1000000;total+=value
            print(row['case_id'],str(value)+' ₽',entry['provider_record']['id'])
before=summary['budget_before']['sol_accounted_micro_usd'];after=summary['budget_after']['sol_accounted_micro_usd']
print('Новые подтверждённые списания:',total,'₽')
print('Рост учёта, включая новые неизвестные резервы:',Decimal(after-before)*89/1000000,'₽')
assert Decimal(after-before)*89/1000000<=Decimal(10)
window_start=config['plan']['budget']['accounted_before_micro_usd']
assert Decimal(after-window_start)*89/1000000<=Decimal(10)
print('Курс 89 ₽/$; старые резервы не обнулены. Mistral бесплатна по условию аккаунта.')

known_error 0.487987 ₽ 124a7577-0827-44ab-9d39-48ebec466a1b
two_questions 0.601996 ₽ f6c42330-161f-4521-acf5-772443429da6
unknown_procedure 0.4717 ₽ 2af3e575-c602-4c42-9131-765d2199cdbd
Новые подтверждённые списания: 1.561683 ₽
Рост учёта, включая новые неизвестные резервы: 1.561683 ₽
Курс 89 ₽/$; старые резервы не обнулены. Mistral бесплатна по условию аккаунта.


In [5]:
expected_generations=0;received=0
for row in rows:
    calls=[a for a in row['attempts'] if not (a.get('diagnostics') or {}).get('not_sent',False)]
    expected_generations+=len(calls)
    if not calls:print(row['case_id'],'остановка до сети — generation не требуется');continue
    proof=json.loads((TASK/row['_evidence_run']/('langfuse-'+row['case_id']+'.json')).read_text(encoding='utf-8'))
    generations=[o for o in proof['observations'] if o['type']=='GENERATION']
    received+=len(generations)
    print(row['case_id'],[(o['name'],o.get('inputUsage'),o.get('outputUsage')) for o in generations])
    assert all(o.get('input') and (o.get('output') or o.get('level')=='ERROR') for o in generations)
print('Generation прочитано:',received,'ожидалось:',expected_generations)
assert received==expected_generations

invoice [('cascade-mistral', 2941, 416)]
notifications [('cascade-mistral', 2946, 442)]
known_error [('cascade-sol', 6136, 329), ('cascade-mistral', 2962, 467)]
two_factor [('cascade-mistral', 2952, 506)]
observer_role [('cascade-mistral', 3188, 478)]
history [('cascade-mistral', 3059, 595)]
missing_code [('cascade-mistral', 3092, 441)]
two_questions [('cascade-sol', 6205, 639), ('cascade-mistral', 3011, 765)]
unknown_procedure [('cascade-sol', 6172, 277), ('cascade-mistral', 3188, 499)]
secret_leak остановка до сети — generation не требуется
Generation прочитано: 12 ожидалось: 12


## Отдельный целевой повтор v3

После двух смысловых ошибок уточнены правила промпта. Это два новых наблюдения на тех же формулировках, а не замена неудачных строк основной серии и не независимая оценка.

In [6]:
follow=TASK/'ml/experiments/cascade/evidence/ordinary-v3-followup-20261110/results.jsonl'
if follow.exists():
    for line in follow.read_text(encoding='utf-8').splitlines():
        r=json.loads(line);old=[json.loads(l) for l in (TASK/'ml/experiments/cascade/evidence/ordinary-v2-live-20261110/results.jsonl').read_text(encoding='utf-8').splitlines()];before=next(x for x in old if x['case_id']==r['case_id'])
        print(r['case_id'],'v2:',before['proposal']['card']['mode'],'v3:',r['proposal']['card']['mode'])
        print(r['proposal']['card']['draft_reply'] or r['proposal']['card']['summary'])
else:print('Целевой повтор не приложен; основная серия не изменена.')

history v2: clarify v3: answer
Ошибка E42 при импорте CSV возникает из-за несоответствия формата файла. Проверьте, что файл сохранён в кодировке UTF-8, и повторите импорт.
two_questions v2: escalate v3: answer
Счёт можно скачать в разделе **Биллинг → Документы**, если у вас есть права администратора аккаунта. 

Чтобы отключить email-уведомления, перейдите в **Личные настройки → Уведомления** и измените настройки согласно вашим предпочтениям.


## Просмотр и ограничения

[Карточки и ссылки на Langfuse](../ml/experiments/cascade/review/ordinary-v2-cards-20261110.html). Оценки человека оставлены пустыми. Совпадение режима не означает доказанную accuracy, а literal provenance не гарантирует смысловую поддержку. Новые проверки снимают лишнюю буквальность служебных полей, не проверку настоящих цитат. Метрики этой простой выборки нельзя напрямую сравнивать с прежним набором из 16 заданий: там были другие входы, 429 и остановки бюджета. Итоги и ограничения — в отчёте ordinary-correctness-20261110.md.